In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics -q
import torch
print('GPU    :', torch.cuda.get_device_name(0))
print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.5 MB/s eta 0:00:00
GPU    : Tesla T4
VRAM   : 15.6 GB


In [2]:
import os

DATASET_ZIP = '/content/drive/MyDrive/lane_detection_dataset/lane_data.zip'
DATASET_OUT = '/content/lane_data'

if not os.path.exists(DATASET_OUT):
    print('Unzipping dataset...')
    !unzip -q "{DATASET_ZIP}" -d /content/
    print('Done.')
else:
    print('Already extracted.')

# Verify
train_imgs = len(os.listdir('/content/lane_data/train/images'))
val_imgs   = len(os.listdir('/content/lane_data/val/images'))
print(f'Train images : {train_imgs}')
print(f'Val images   : {val_imgs}')

Unzipping dataset...
Done.
Train images : 2901
Val images   : 362


In [3]:
yaml_content = """
path: /content/lane_data
train: train/images
val:   val/images
test:  test/images
nc: 1
names:
  0: lane
"""
with open('/content/lane_data.yaml', 'w') as f:
    f.write(yaml_content)
print('YAML written.')

YAML written.


In [ ]:
from ultralytics import YOLO
import os
import shutil

#Paths
DRIVE_PROJECT = '/content/drive/MyDrive/lane_detection_dataset/runs'
RUN_NAME      = 'lane_seg_v1'
LAST_PT       = f'{DRIVE_PROJECT}/{RUN_NAME}/weights/last.pt'
BEST_PT       = f'{DRIVE_PROJECT}/{RUN_NAME}/weights/best.pt'

if os.path.exists(LAST_PT):
    print(f'Resuming from checkpoint: {LAST_PT}')
    model  = YOLO(LAST_PT)
    resume = True
else:
    print('Starting fresh training...')
    model  = YOLO('yolov8s-seg.pt')
    resume = False

results = model.train(
    data    = '/content/lane_data.yaml',
    task    = 'segment',

    epochs  = 100,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    workers = 4,
    amp     = True,

    # Seg-specific
    overlap_mask = False,
    mask_ratio   = 1,
    retina_masks = True,

    # Augmentation
    flipud   = 0.0,
    fliplr   = 0.5,
    degrees  = 5.0,
    shear    = 2.0,
    perspective = 0.0003,
    hsv_h    = 0.015,
    hsv_s    = 0.5,
    hsv_v    = 0.4,
    mosaic   = 1.0,
    close_mosaic = 15,
    copy_paste = 0.1,
    erasing  = 0.0,
    mixup    = 0.0,

    #Optimizer
    optimizer     = 'SGD',
    lr0           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3.0,
    patience      = 30,

    #Saving
    save        = True,
    save_period = 4,
    plots       = True,

    project  = DRIVE_PROJECT,
    name     = RUN_NAME,
    exist_ok = True,
    resume   = resume,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting fresh training...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/lane_data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=

In [ ]:
# Continution after 52 epoch — fine-tuning phase
from ultralytics import YOLO
import os

DRIVE_PROJECT = '/content/drive/MyDrive/lane_detection_dataset/runs'
RUN_NAME      = 'lane_seg_v2'
LAST_PT       = f'{DRIVE_PROJECT}/lane_seg_v1/weights/last.pt'

print('Loading weights from v1, fine-tuning with lower LR...')
model = YOLO(LAST_PT)

model.train(
    data    = '/content/lane_data.yaml',
    task    = 'segment',

    epochs  = 50,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    workers = 4,
    amp     = True,

    # Seg-specific
    overlap_mask = False,
    mask_ratio   = 1,
    retina_masks = True,

    #Key changes for fine-tuning
    lr0          = 0.001,
    lrf          = 0.01,
    warmup_epochs= 0,

    # Reduced augmentation for fine-tuning
    mosaic       = 0.3,
    close_mosaic = 5,
    copy_paste   = 0.0,
    degrees      = 2.0,
    shear        = 1.0,

    flipud       = 0.0,
    fliplr       = 0.5,
    hsv_h        = 0.015,
    hsv_s        = 0.3,
    hsv_v        = 0.3,

    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    patience      = 15,

    save        = True,
    save_period = 4,
    plots       = True,
    resume      = False,

    project  = DRIVE_PROJECT,
    name     = RUN_NAME,
    exist_ok = True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading weights from v1, fine-tuning with lower LR...
Ultralytics 8.4.34 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/lane_data.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f44b4188aa0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

Fine Tuning Phase 3

In [ ]:
from ultralytics import YOLO
import os

# Paths
DRIVE_PROJECT = '/content/drive/MyDrive/lane_detection_dataset/runs'
PHASE2_BEST   = f'{DRIVE_PROJECT}/lane_seg_v2/weights/best.pt'
RUN_NAME      = 'lane_seg_v3_elite'

print('--- Phase 3: High-Resolution Precision Refinement ---')
model = YOLO(PHASE2_BEST)

results = model.train(
    data    = '/content/lane_data.yaml',
    task    = 'segment',

    #  The Resolution Boost
    imgsz   = 800,
    epochs  = 30,
    batch   = 8,

    # Ultra-Low Learning Rate
    lr0           = 0.0001,
    lrf           = 0.01,
    warmup_epochs = 0,
    optimizer     = 'AdamW',

    #ZERO Augmentations
    mosaic       = 0.0,
    mixup        = 0.0,
    copy_paste   = 0.0,
    degrees      = 0.0,
    perspective  = 0.0,
    scale        = 0.1,
    flipud       = 0.0,
    fliplr       = 0.5,

    #  Segmentation Settings
    retina_masks = True,
    mask_ratio   = 1,
    overlap_mask = False,

    # Saving & Checkpoints
    save         = True,
    save_period  = 4,
    project      = DRIVE_PROJECT,
    name         = RUN_NAME,
    exist_ok     = True,

    # Performance
    device       = 0,
    amp          = True,
    plots        = True
)

--- Phase 3: High-Resolution Precision Refinement ---
Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/lane_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=1, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/lane_detection_dataset/runs/lane_seg_v2/weights/best.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=l

In [ ]:
import shutil
from pathlib import Path

DRIVE_PROJECT = '/content/drive/MyDrive/lane_detection_dataset/runs'
RUN_NAME      = 'lane_seg_v2'
WEIGHTS_DIR   = Path(f'{DRIVE_PROJECT}/{RUN_NAME}/weights')
BACKUP_DIR    = Path(f'{DRIVE_PROJECT}/{RUN_NAME}/backups')
BACKUP_DIR.mkdir(exist_ok=True)

last = WEIGHTS_DIR / 'last.pt'
best = WEIGHTS_DIR / 'best.pt'

if last.exists():
    import torch
    # Fix 1: map_location='cpu' not 'gpu'
    # Fix 2: weights_only=False needed for Ultralytics checkpoints
    ckpt  = torch.load(str(last), map_location='cpu', weights_only=False)
    epoch = ckpt.get('epoch', 'unknown')
    shutil.copy2(str(last), str(BACKUP_DIR / f'checkpoint_epoch{epoch}.pt'))
    print(f'Backed up last.pt at epoch {epoch}')
else:
    print('last.pt not found')

if best.exists():
    shutil.copy2(str(best), str(BACKUP_DIR / 'best_v2_latest.pt'))
    print('Backed up best.pt')
else:
    print('best.pt not found')

print('\nBackup contents:')
for f in sorted(BACKUP_DIR.glob('*.pt')):
    print(f'  {f.name} — {f.stat().st_size/1e6:.1f} MB')

Backed up last.pt at epoch 1
Backed up best.pt

Backup contents:
  best_v2_latest.pt — 47.5 MB
  checkpoint_epoch1.pt — 47.5 MB


In [4]:
from ultralytics import YOLO

model   = YOLO("/content/drive/MyDrive/lane_detection_dataset/runs/lane_seg_v2/weights/best.pt")
metrics = model.val(data='/content/lane_data.yaml')

print(f'\nBox  mAP50    : {metrics.box.map50:.3f}')
print(f'Box  mAP50-95 : {metrics.box.map:.3f}')
print(f'Mask mAP50    : {metrics.seg.map50:.3f}')
print(f'Mask mAP50-95 : {metrics.seg.map:.3f}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1741.0±902.3 MB/s, size: 152.5 KB)
val: Scanning /content/lane_data/val/labels... 362 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 362/362 1.2Kit/s 0.3s
val: New cache created: /content/lane_data/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        362       1271      0.913      0.941      0.965